# deltaSVM Pipeline
End-to-end pipeline for scoring SNP allelic effects on TF binding using deltaSVM.

## 1. Prepare SNPs from dbSNP VCF (`prepare_snps.py`)

In [ ]:
import gzip

input_vcf = "/mnt/hdd_1/abdu_md/biocypher_data_bizon/dbsnp/00-common_all.vcf.gz"
output_snp = "input_snp_common.tsv"
output_rsid_map = "rsid_map.tsv"

count = 0
written = 0

with gzip.open(input_vcf, "rt") as f, \
     open(output_snp, "w") as out_snp, \
     open(output_rsid_map, "w") as out_map:

    out_map.write("variant\trsId\n")

    for line in f:
        if line.startswith("#"):
            continue

        cols = line.strip().split("\t")
        chrom, pos, rsid, ref, alt, info = cols[0], cols[1], cols[2], cols[3], cols[4], cols[7]

        count += 1
        if "VC=SNV" not in info:
            continue
        if chrom not in [str(i) for i in range(1, 23)] + ["X", "Y"]:
            continue
        if len(ref) != 1 or len(alt) != 1:
            continue

        variant = f"chr{chrom}_{pos}_{ref}_{alt}"
        out_snp.write(variant + "\n")
        out_map.write(f"{variant}\t{rsid}\n")
        written += 1

        if written % 1000000 == 0:
            print(f"Processed {count:,} lines, written {written:,} SNPs...")

print(f"Done! Total lines: {count:,}, SNPs written: {written:,}")

## 2. Filter SNPs Using OpenTargets Variants (`filter_ot_snps.py`)

In [ ]:
import pandas as pd
import glob

print("Loading OpenTargets variants...")
files = glob.glob('/mnt/hdd_1/rediet/opentargets_variants/variant/part-*.parquet')
ot_variants = set()
for f in files:
    df = pd.read_parquet(f, columns=['variantId'])
    df['variantId'] = 'chr' + df['variantId']
    ot_variants.update(df['variantId'].tolist())
    print(f"Loaded {f.split('/')[-1]}: {len(ot_variants):,} variants so far")

print(f"\nTotal OT variants: {len(ot_variants):,}")

for CHR in ['chr4', 'chr5', 'chr6', 'chr7', 'chr8', 'chr9']:
    snp_file = f'/mnt/hdd_1/rediet/deltaSVM/snp_batches/{CHR}.tsv'
    out_file = f'/mnt/hdd_1/rediet/deltaSVM/snp_batches/{CHR}_ot.tsv'

    with open(snp_file) as f:
        snps = [line.strip() for line in f]

    filtered = [s for s in snps if s in ot_variants]

    with open(out_file, 'w') as f:
        f.write('\n'.join(filtered))

    print(f"{CHR}: {len(snps):,} → {len(filtered):,} SNPs after filtering")

## 3. Set Up Per-Chromosome Working Directory (`setup_chr.sh`)

In [ ]:
%%bash
CHR=$1
BASE_DIR="/mnt/hdd_1/rediet/deltaSVM"

echo "Setting up $CHR..."
mkdir -p $BASE_DIR/runs/$CHR/{data,tmp,out,log}
cp -r $BASE_DIR/scripts $BASE_DIR/runs/$CHR/
cp -r $BASE_DIR/resources $BASE_DIR/runs/$CHR/
cp -r $BASE_DIR/gkmsvm_models $BASE_DIR/runs/$CHR/
cp $BASE_DIR/snp_batches/$CHR.tsv $BASE_DIR/runs/$CHR/input_snp.tsv

cd $BASE_DIR/runs/$CHR
python scripts/generate_allelic_seqs.py -f resources/hs38/hs38.fa -s input_snp.tsv -o data/selex_allelic_oligos
scripts/deltasvm_subset_multi data/selex_allelic_oligos.ref.fa data/selex_allelic_oligos.alt.fa resources/models.weights.txt out/pbs.pred.tsv resources/thresholds.pbs.tsv
echo "$CHR setup done!"